# Behind the scenes: Day Two
### Data download and preparation

1. Beck et al. ALREADY DONE IN DAY1_BTS!
2. GBIF species occurrence records
3. Sentinel-2 satellite imagery for Negros island
4. Hansen Global Forest Change data for Negros

In [ ]:
from pathlib import Path
import re
import requests
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import rioxarray
from rasterio.windows import from_bounds
from rasterio.io import MemoryFile
from rasterio.merge import merge as rio_merge
from pyproj import Transformer
import pystac_client

from workshop_utils import RAW_DIR, PROCESSED_DIR
print(f'RAW_DIR: {RAW_DIR}')

---
## 1. Beck et al. (2023) — Köppen-Geiger maps and 1 km climate data

> Beck, H. E., et al. (2023). High-resolution (1 km) Köppen-Geiger maps

Already done!

---
## 2. GBIF species occurrence records

Two species, two purposes:
- **Philippine Eagle** (*Pithecophaga jefferyi*, GBIF taxonKey 2480965) — Day 2 biodiversity showcase.
  Even with few records, the map tells a powerful conservation story.
- **Philippine Flying Fox** (*Pteropus vampyrus*, GBIF taxonKey 2432859) — Day 3 SDM target.
  Widespread across the archipelago, with enough records for a meaningful model.

No API key required. Pagination via `offset`.

In [ ]:
def download_gbif(taxon_key, country='PH', out_path=None):
    records, offset, limit = [], 0, 300
    base = 'https://api.gbif.org/v1/occurrence/search'
    while True:
        r = requests.get(base, params={
            'taxonKey': taxon_key, 'country': country,
            'hasCoordinate': 'true', 'hasGeospatialIssue': 'false',
            'limit': limit, 'offset': offset,
        })
        r.raise_for_status()
        data = r.json()
        records.extend(data['results'])
        print(f"  {len(records)}/{data['count']}", end='\r')
        if data['endOfRecords']:
            break
        offset += limit
    print()

    df = pd.DataFrame([
        {
            'species':  rec.get('species', ''),
            'lon':      rec.get('decimalLongitude'),
            'lat':      rec.get('decimalLatitude'),
            'year':     rec.get('year'),
            'basis':    rec.get('basisOfRecord', ''),
            'gbifID':   rec.get('gbifID'),
        }
        for rec in records if rec.get('decimalLongitude') is not None
    ])

    if out_path is not None:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_path, index=False)
        print(f'  saved {len(df)} records -> {out_path}')
    return df

In [ ]:
eagle = download_gbif(
    taxon_key=2480965,
    out_path=RAW_DIR / 'GBIF' / 'philippine_eagle.csv',
)
eagle.head()

In [ ]:
fox = download_gbif(
    taxon_key=2432859,
    out_path=RAW_DIR / 'GBIF' / 'flying_fox.csv',
)
print(fox['year'].value_counts().sort_index().tail(15))
fox.head()

---
## 3. Sentinel-2 satellite imagery — Negros island

Two low-cloud scenes (historical ~2017, recent ~2023-2024) over central Negros.
Data from the **Element84 Earth Search** STAC catalog — no credentials needed.

Bands:
- `red` (B04, 10 m) + `nir` (B08, 10 m) — for NDVI
- `visual` (true-colour TCI, 10 m) — for display

Output: `data/raw/Sentinel2/negros_s2.nc`  with dimensions `(time, latitude, longitude)`.

In [ ]:
NEGROS_BBOX = [122.3, 9.8, 123.4, 11.0]   # central Negros, [W, S, E, N]

catalog = pystac_client.Client.open('https://earth-search.aws.element84.com/v1')

def search_s2(date_range, cloud_max=20):
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=NEGROS_BBOX,
        datetime=date_range,
        query={'eo:cloud_cover': {'lt': cloud_max}},
        max_items=20,
    )
    items = list(search.items())
    print(f'  {len(items)} scenes found:')
    for it in sorted(items, key=lambda x: x.properties.get('eo:cloud_cover', 99)):
        cc = it.properties.get('eo:cloud_cover', '?')
        dt = it.properties.get('datetime', '')[:10]
        print(f'    {it.id}   {dt}   cloud={cc:.1f}%')
    return items

print('Searching 2016-2018 (historical):')
items_old = search_s2('2016-01-01/2018-12-31')

print('\nSearching 2023-2024 (recent):')
items_new = search_s2('2023-01-01/2024-12-31')

In [ ]:
def read_band_cropped(href, bbox_wgs84):
    """Read a COG band (remote or local), crop to WGS84 bbox, return array + metadata."""
    with rasterio.open(href) as src:
        if src.crs.to_epsg() != 4326:
            t = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
            xmin, ymin = t.transform(bbox_wgs84[0], bbox_wgs84[1])
            xmax, ymax = t.transform(bbox_wgs84[2], bbox_wgs84[3])
        else:
            xmin, ymin, xmax, ymax = bbox_wgs84

        win = from_bounds(xmin, ymin, xmax, ymax, src.transform)
        row0 = max(0, int(win.row_off))
        col0 = max(0, int(win.col_off))
        row1 = min(src.height, int(win.row_off + win.height))
        col1 = min(src.width,  int(win.col_off + win.width))
        win_c = rasterio.windows.Window(col0, row0, col1 - col0, row1 - row0)

        data = src.read(window=win_c)
        transform = src.window_transform(win_c)
        profile = src.profile.copy()
        profile.update(
            width=data.shape[2], height=data.shape[1],
            transform=transform, crs=src.crs, count=data.shape[0],
        )
    return data, transform, profile


def download_s2_scene(item, label, target_bands=('red', 'nir', 'visual')):
    out_dir = RAW_DIR / 'Sentinel2' / label
    out_dir.mkdir(parents=True, exist_ok=True)

    # Asset key fallbacks across different STAC versions
    fallbacks = {'nir': ['nir', 'nir08', 'B08'], 'red': ['red', 'B04'], 'visual': ['visual', 'tci']}

    for band in target_bands:
        out_path = out_dir / f'{band}.tif'
        if out_path.exists():
            print(f'  skip {label}/{band}')
            continue

        asset_key = next((k for k in fallbacks.get(band, [band]) if k in item.assets), None)
        if asset_key is None:
            print(f'  WARNING: {band!r} not found; available: {list(item.assets.keys())}')
            continue

        href = item.assets[asset_key].href
        print(f'  {label}/{band} ...', end='', flush=True)
        data, _, profile = read_band_cropped(href, NEGROS_BBOX)
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)
        print(f' {data.shape}')


best_old = sorted(items_old, key=lambda x: x.properties.get('eo:cloud_cover', 999))[0]
best_new = sorted(items_new, key=lambda x: x.properties.get('eo:cloud_cover', 999))[0]

print(f'Old scene : {best_old.id}  ({best_old.properties["datetime"][:10]})')
print(f'New scene : {best_new.id}  ({best_new.properties["datetime"][:10]})\n')

download_s2_scene(best_old, 'old')
download_s2_scene(best_new, 'new')

In [ ]:
# Combine into a single NetCDF with a 'time' dimension
old_date = best_old.properties['datetime'][:10]
new_date = best_new.properties['datetime'][:10]

def load_s2_slice(label, date):
    tif_dir = RAW_DIR / 'Sentinel2' / label
    da_dict = {}
    for tif_path in sorted(tif_dir.glob('*.tif')):
        band = tif_path.stem
        da = rioxarray.open_rasterio(tif_path, masked=True)
        da = da.rename({'x': 'longitude', 'y': 'latitude'})
        if 'band' in da.dims and da.sizes['band'] == 1:
            da = da.squeeze('band', drop=True)
        da_dict[band] = da
    return xr.Dataset(da_dict).expand_dims(time=[pd.Timestamp(date)])

ds_s2 = xr.concat(
    [load_s2_slice('old', old_date), load_s2_slice('new', new_date)],
    dim='time',
)
out_nc = RAW_DIR / 'Sentinel2' / 'negros_s2.nc'
ds_s2.to_netcdf(out_nc)
print(f'Saved: {out_nc}')
print(ds_s2)

---
## 4. Hansen Global Forest Change — Negros

Hansen et al. (2013) provide annual forest cover and loss data from 2000 to present
at 30 m resolution as public COGs on Google Cloud Storage.

Layers:
- `treecover2000` — canopy cover in year 2000 (%)
- `lossyear` — year of first forest loss (1=2001 … 23=2023, 0=no loss)

Negros (9–11°N) straddles two 10°×10° tiles (`20N_120E` and `10N_120E`).
We use rasterio windowed reads on the remote COGs — only the Negros-sized patch
is transferred, not the full global tile.

In [ ]:
HANSEN_BASE   = 'https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11'
HANSEN_LAYERS = ['treecover2000', 'lossyear']
HANSEN_TILES  = ['20N_120E', '10N_120E']   # northern tile first (higher latitude)
NEGROS_FULL   = (121.5, 8.8, 124.5, 11.5)  # W, S, E, N — slightly wider than NEGROS_BBOX

out_dir = RAW_DIR / 'Hansen'
out_dir.mkdir(parents=True, exist_ok=True)

for layer in HANSEN_LAYERS:
    out_path = out_dir / f'hansen_{layer}_negros.tif'
    if out_path.exists():
        print(f'skip {out_path.name}')
        continue

    print(f'{layer}:')
    mem_files = []
    for tile in HANSEN_TILES:
        url = f'{HANSEN_BASE}/Hansen_GFC-2023-v1.11_{layer}_{tile}.tif'
        print(f'  reading {tile} ...', end='', flush=True)
        try:
            with rasterio.open(url) as src:
                win = from_bounds(*NEGROS_FULL, src.transform)
                row0 = max(0, int(win.row_off))
                col0 = max(0, int(win.col_off))
                row1 = min(src.height, int(win.row_off + win.height))
                col1 = min(src.width,  int(win.col_off + win.width))
                if row1 <= row0 or col1 <= col0:
                    print(' no overlap — skip')
                    continue
                win_c = rasterio.windows.Window(col0, row0, col1 - col0, row1 - row0)
                data = src.read(1, window=win_c)
                transform = src.window_transform(win_c)
                profile = src.profile.copy()
                profile.update(
                    width=data.shape[1], height=data.shape[0],
                    transform=transform, count=1,
                )
            mem = MemoryFile()
            with mem.open(**profile) as mds:
                mds.write(data, 1)
            mem_files.append(mem)
            print(f' {data.shape}')
        except Exception as e:
            print(f' ERROR: {e}')

    if not mem_files:
        print(f'  no data for {layer}')
        continue

    opened = [m.open() for m in mem_files]
    mosaic, out_transform = rio_merge(opened)
    profile_out = opened[0].profile.copy()
    profile_out.update(
        height=mosaic.shape[1], width=mosaic.shape[2],
        transform=out_transform, count=1,
    )
    for d in opened:
        d.close()
    for m in mem_files:
        m.close()

    with rasterio.open(out_path, 'w', **profile_out) as dst:
        dst.write(mosaic[0], 1)

    print(f'  -> saved {out_path.name}  {mosaic.shape}')

print('\nHansen download complete.')